# DETECTING EARLY SIGNALS

In this file we are going to analize how our variables behave related to the target variable, so that we can detect pattrons that can serve as early signals of the beginning of a new allocation-elegible period of a country.

In [37]:
import pandas as pd

In [38]:
df = pd.read_csv("../data_clean/complete_dataset.csv")

In [39]:
df.head()

,iso3,month,risk_3,risk_12,logfat_risk_3,logfat_risk_12,INFORM,VU,CC,HA,...,event_count,notes_acled,hdx_alert_level,hdx_value,monthly_displacement,inform_severity_index,rolling_3m_displacements,allocation-eligible,target_2m,start_conflict
0,AFG,2019-01-01,0.997366,1.000000,8.554694,9.716417,7.7,7.1,7.5,8.7,...,31.0,"[""On 1 January 2019, 10 Taliban militants were...",[],0.000000,10789.172972,4.133333,10789.172972,0,1.0,0
1,AFG,2019-02-01,0.993118,0.997806,8.489241,9.786673,7.7,7.1,7.5,8.7,...,28.0,"[""On 01 February 2019, 1 policeman was killed ...",[],0.000000,9745.059458,4.400000,20534.232430,0,1.0,0
2,AFG,2019-03-01,0.994183,0.997635,8.551225,9.668575,7.7,7.1,7.5,8.7,...,31.0,"[""As reported on March 2, over 24 hours, Afgha...",['Medium concern'],35111.111111,66039.172976,3.250000,86573.405407,1,NaN,1
3,AFG,2019-04-01,0.992014,0.997427,8.497727,9.726764,7.7,7.1,7.5,8.7,...,30.0,"[""Detonation: On 01-April-2019, 1 Taliban mili...",[],0.000000,15691.135135,3.350000,91475.367569,1,NaN,0
4,AFG,2019-05-01,0.993915,0.998244,8.677351,9.830026,7.8,7.2,7.5,8.8,...,31.0,"['As reported on 01-May-2019, 19 Taliban milit...",[],0.000000,10789.172972,4.500000,92519.481083,1,1.0,0


First we are going to check all all the datasets separately, via plots, so that is more comprehensive.

In [42]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

def plot_full_crisis_timeline(df, country_code, columns_to_plot, cerf_path=None):
    
    # 1. Limpieza de seguridad por espacios invisibles en los nombres
    df.columns = df.columns.str.strip()
    columns_to_plot = [c.strip() for c in columns_to_plot]
    
    # --- DICCIONARIO DE CONFIGURACIÓN DE UNIDADES ---
    COL_CONFIG = {
        # --- Paquete Absolute (Eje Y1) ---
        'fatalities': {'name': 'Fatalities', 'color': 'orange', 'unit': 'absolute', 'dash': 'solid'},
        
        # --- Paquete Log (Eje Y2) ---
        'event_count': {'name': 'Event Count (Log)', 'color': 'teal', 'unit': 'log', 'dash': 'solid', 'log_transform': True},
        'hdx_value': {'name': 'HDX Value (Log)', 'color': 'blue', 'unit': 'log', 'dash': 'solid', 'log_transform': True},
        'monthly_displacement': {'name': 'Displacements (Log)', 'color': 'purple', 'unit': 'log', 'dash': 'solid', 'log_transform': True},
        'rolling_3m_displacements': {'name': 'Rolling Disp 3m (Log)', 'color': 'magenta', 'unit': 'log', 'dash': 'dash', 'log_transform': True},
        'logfat_risk_3': {'name': 'LogFat Risk 3m', 'color': 'darkorange', 'unit': 'log', 'dash': 'dot'},
        'logfat_risk_12': {'name': 'LogFat Risk 12m', 'color': 'peru', 'unit': 'log', 'dash': 'dot'},
        
        # --- Paquete Index (Eje Y3) ---
        'risk_3': {'name': 'Risk 3m', 'color': 'black', 'unit': 'index', 'dash': 'dash'},
        'risk_12': {'name': 'Risk 12m', 'color': 'grey', 'unit': 'index', 'dash': 'dash'},
        'INFORM': {'name': 'INFORM Score', 'color': 'darkgreen', 'unit': 'index', 'dash': 'solid'},
        'VU': {'name': 'Vulnerability (VU)', 'color': 'darkred', 'unit': 'index', 'dash': 'solid'},
        'CC': {'name': 'Coping Capacity (CC)', 'color': 'blue', 'unit': 'index', 'dash': 'solid'},
        'HA': {'name': 'Hazards & Exposure (HA)', 'color': 'red', 'unit': 'index', 'dash': 'solid'},
        'inform_severity_index': {'name': 'INFORM Severity', 'color': 'olive', 'unit': 'index', 'dash': 'solid'},
    }

    UNIT_TO_AXIS = {'absolute': 'y1', 'log': 'y2', 'index': 'y3'}

    # Filtrar el DataFrame por país
    df_plot = df[df['iso3'] == country_code].copy()
    if df_plot.empty:
        print(f"⚠️ No hay datos en el DataFrame para el código de país: {country_code}")
        return
        
    df_plot['month'] = pd.to_datetime(df_plot['month'])
    df_plot = df_plot.sort_values('month')

    # Cargar datos de CERF si se provee la ruta
    cerf = pd.DataFrame()
    if cerf_path:
        try:
            cerf = pd.read_csv(cerf_path)
            cerf = cerf[cerf['iso3'] == country_code].copy()
            cerf['Allocation Date'] = pd.to_datetime(cerf['Allocation Date'], format='ISO8601', errors='coerce')
            cerf = cerf.dropna(subset=['Allocation Date'])
        except FileNotFoundError:
            print(f"⚠️ No se encontró el archivo CERF en {cerf_path}.")

    # Detectar qué unidades matemáticas se van a pintar
    valid_columns = [col for col in columns_to_plot if col in df_plot.columns and col in COL_CONFIG]
    
    # Forzar activación del eje 'index' si se pide hdx_alert_level de manera explícita
    active_units = {COL_CONFIG[col]['unit'] for col in valid_columns}
    if 'hdx_alert_level' in columns_to_plot and 'hdx_alert_level' in df_plot.columns:
        active_units.add('index')

    fig = go.Figure()

    # --- 2. DIBUJAR TRAZAS DE LÍNEAS REGULARES ---
    for col in valid_columns:
        config = COL_CONFIG[col]
        axis_id = UNIT_TO_AXIS[config['unit']]
        
        series_data = pd.to_numeric(df_plot[col], errors='coerce')
        
        if config.get('log_transform'):
            y_data = np.log1p(series_data.fillna(0))
            hover = f'<b>Date:</b> %{{x|%b %Y}}<br><b>{config["name"]} (Real):</b> %{{customdata:,.0f}}<br><b>Log(1+x):</b> %{{y:.2f}}<extra></extra>'
            custom = series_data
        else:
            y_data = series_data
            # Un ajuste dinámico al hover para que muestre enteros limpios si es conteo o fatalities
            fmt = ',.0f' if config['unit'] == 'absolute' else '.2f'
            hover = f'<b>Date:</b> %{{x|%b %Y}}<br><b>{config["name"]}:</b> %{{y:{fmt}}}<extra></extra>'
            custom = None

        fig.add_trace(go.Scatter(
            x=df_plot['month'], y=y_data, customdata=custom, mode='lines',
            name=config['name'], line=dict(color=config['color'], width=2.5, dash=config['dash']),
            yaxis=axis_id, hovertemplate=hover
        ))

    # --- 3. LÓGICA DE ALERTAS HDX (TEXTO A PUNTOS FLOTANTES) ---
    if 'hdx_alert_level' in columns_to_plot and 'hdx_alert_level' in df_plot.columns:
        hdx_points = []
        for _, row in df_plot.iterrows():
            alerts = str(row['hdx_alert_level']).lower()
            if 'high concern' in alerts:
                hdx_points.append({'month': row['month'], 'level': 'High concern', 'color': 'red', 'y': 1.08})
            if 'medium concern' in alerts:
                hdx_points.append({'month': row['month'], 'level': 'Medium concern', 'color': 'gold', 'y': 1.03})
        
        hdx_alerts = pd.DataFrame(hdx_points)
        
        if not hdx_alerts.empty:
            fig.add_trace(go.Scatter(
                x=hdx_alerts['month'], y=hdx_alerts['y'], mode='markers', name='HDX Alerts',
                marker=dict(size=8, color=hdx_alerts['color'], symbol='circle', line=dict(width=0.5, color='black')),
                yaxis='y3', customdata=hdx_alerts['level'],
                hovertemplate='<b>HDX SIGNAL</b><br>Date: %{x|%b %Y}<br>Level: %{customdata}<extra></extra>'
            ))

    # --- 4. DIBUJAR ELEMENTOS FIJOS DE FONDO ---
    if 'allocation-eligible' in df_plot.columns:
        for c_month in df_plot[df_plot['allocation-eligible'] == 1]['month']:
            fig.add_vrect(
                x0=c_month - pd.Timedelta(days=15), x1=c_month + pd.Timedelta(days=15),
                fillcolor="salmon", opacity=0.15, layer="below", line_width=0, y0=0, y1=1
            )
        fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines', name='Allocation-eligible', line=dict(color='salmon', width=10), opacity=0.3))
    
    if 'target_2m' in df_plot.columns:
        for w_date in df_plot[df_plot['target_2m'] == 1]['month']:
            fig.add_shape(type="line", x0=w_date, x1=w_date, y0=0, y1=1, xref="x", yref="paper", line=dict(width=2, dash="dash", color="red"))
        fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines', name='Warning (target_2m)', line=dict(color='red', width=2, dash='dash')))

    if not cerf.empty:
        for _, row in cerf.iterrows():
            fig.add_shape(type="line", x0=row['Allocation Date'], x1=row['Allocation Date'], y0=0, y1=1, xref="x", yref="paper", line=dict(width=2.5, dash="dot", color="mediumseagreen"))
        fig.add_trace(go.Scatter(x=[None], y=[None], mode='lines', name='CERF Allocation', line=dict(color='mediumseagreen', width=2.5, dash='dot')))

    # --- 5. MAQUETADO DINÁMICO DE EJES ---
    needs_extra_margin = ('log' in active_units and 'index' in active_units)
    
    layout_update = dict(
        title=dict(text=f'<b>Crisis Timeline Overview: {country_code}</b>', font=dict(size=20), x=0.05),
        margin=dict(l=60, r=60, t=100, b=60), height=600, plot_bgcolor='white', hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5, bgcolor='rgba(255,255,255,0.8)'),
        
        xaxis=dict(domain=[0, 0.85] if needs_extra_margin else [0, 1.0], title="<b>Date</b>", showgrid=False, tickformat="%b %Y", tickangle=45),
        yaxis=dict(title="<b>Absolute Scale (Count / Fatalities)</b>" if 'absolute' in active_units else "", showgrid=True, gridcolor='lightgrey', side="left")
    )

    if 'log' in active_units:
        layout_update['yaxis2'] = dict(title="<b>Log Scale</b>", tickfont=dict(color="purple"), anchor="x", overlaying="y", side="right", showgrid=False)

    if 'index' in active_units:
        pos = 0.95 if needs_extra_margin else 1.0
        
        has_inform_cols = any(x in valid_columns for x in ['INFORM', 'VU', 'CC', 'HA', 'inform_severity_index'])
        y3_range = [0, 10.5] if has_inform_cols else [0, 1.25]
        y3_title = "<b>INFORM Scale (0-10)</b>" if has_inform_cols else "<b>Index / Alert Scale</b>"
        
        layout_update['yaxis3'] = dict(
            title=y3_title, anchor="free" if needs_extra_margin else "x", 
            overlaying="y", side="right", position=pos, range=y3_range, showgrid=False
        )

    fig.update_layout(**layout_update)
    fig.show()

In [43]:
plot_full_crisis_timeline(
    df, 
    country_code="SYR",
    columns_to_plot=['fatalities', 'event_count']
)

In [44]:
plot_full_crisis_timeline(
    df, 
    country_code="SYR",
    columns_to_plot=['monthly_displacement']
)

In [45]:
plot_full_crisis_timeline(
    df, 
    country_code="SYR",
    columns_to_plot=["risk_3", "risk_12", "logfat_risk_3", "logfat_risk_12"]
)

In [46]:
plot_full_crisis_timeline(
    df, 
    country_code="SYR",
    columns_to_plot=["INFORM", "VU", "CC", "HA"]
)

In [47]:
plot_full_crisis_timeline(
    df, 
    country_code="SYR",
    columns_to_plot=["hdx_alert_level", "hdx_value"]
)

In [48]:
plot_full_crisis_timeline(
    df, 
    country_code="SYR",
    columns_to_plot=["inform_severity_index"]
)